# Import the necessary libraries and modules

In [ ]:
import torch  # PyTorch library for deep learning
import torch.optim as optim  # Optimizers for gradient-based optimization
import torch.nn as nn  # Neural network modules and loss functions
import copy  # Used for creating deep copies of objects
import time  # Provides time-related functionality
import pennylane as qml  # PennyLane for quantum machine learning
import numpy as np  # NumPy library for numerical computations
from torch.utils.data import DataLoader  # DataLoader for handling data batches
dtype = torch.float  # Set default tensor data type to float
seed = 42

In [ ]:
def Solution(t, params):
    """
    Computes the exact values of solution.

    Args:
        t (torch.Tensor): Distance from the gravitational source.
        params (list): List of parameter contains the mass of the gravitational source.

    Returns:
        torch.Tensor: component g00 values respect to mass.
    """
    A, omega, beta = params
    return A*torch.exp(-beta*t)*torch.cos(np.sqrt((omega**2) - (beta**2))*t)

def grad(outputs, inputs):
    """
    Computes the gradient of outputs with respect to inputs.

    Args:
        outputs (torch.Tensor): Output tensor.
        inputs (torch.Tensor): Input tensor.

    Returns:
        Gradient tensor.
    """
    return torch.autograd.grad(outputs, inputs, grad_outputs=torch.ones_like(outputs), create_graph=True)[0]

def phy_loss(t, y, dy_dt, d2y_dt2, params):
    """
    Computes the physics-informed loss function (physics experssion).

    Args:
        t (torch.Tensor): Time.
        y (torch.Tensor): function predicted by neural network.
        dy (torch.Tensor): First derivative of the predicted function with respect to t.
        d2y_dt2 (torch.Tensor): Second derivative of the predicted time with respect to t.
        params (torch.Tensor): List containing amplitude, angular frequency, damping parameter.
    Returns:
        torch.Tensor: Mean of physics-informed loss function.
    """
    A, omega, beta = params
    return torch.mean((d2y_dt2 + 2*beta*dy_dt + (omega**2)*y)**2)

def bc1_loss(y0, y0p):
    """
    Computes the loss based on the first boundary condition: y(t0) = y0.

    Args:
        y0 (torch.Tensor): The exact answer (ground truth) at the initial time point t0.
        y0p (torch.Tensor): The predicted answer by the neural network at t0.
        
    Returns:
        torch.Tensor: Mean of first boundary condition loss function.
    """
    return torch.mean((y0 - y0p)**2)

def bc2_loss(dy0_dt, dy0p_dt):
    """
    Computes the loss based on the second boundary condition: dy/dt at t0 = dy0_dt.

    Args:
        dy0_dt (torch.Tensor): The exact derivative of y with respect to t at the initial time point t0.
        dy0p_dt (torch.Tensor): The predicted derivative by the neural network at t0.
        
    Returns:
        torch.Tensor: Mean of second boundary condition loss function.
    """
    return torch.mean((dy0_dt - dy0p_dt)**2)

def train_PINN(model, batches, batch_size, domain, params ,epochs, num_itr, loss_weights, seed, save_model=[False, "model"]):
    """
    Trains a physics-informed neural network (PINN) model.

    Args:
        model: The neural network model.
        batches (int): Number of points in the domain.
        batch_size (int): Number of points in each batch.
        domain (list): Defined domain in the interval [0, 8].
        params (list): List containing amplitude, angular frequency, damping parameter.
        epochs (int): Number of epochs.
        num_itr (int): Number of iteration.
        loss_weights (list): List containing weights of overall loss function.
        save_model (list): Optional flag for saving the model.

    Returns:
        Trained model, Loss function and its components.
    """
    # Initialize the best trained neural network model (bNN) as None
    torch.manual_seed(seed)
    bNN = None
    # Initialize total loss (tot_l) as None
    tot_l = None
    
    # Initialize empty lists for different loss histories
    phy_loss_history = [] # physics-informed loss function
    bc1_loss_history = [] # Boundary condition 1 loss functoin
    bc2_loss_history = [] # Boundary condition 2 loss function
    loss_history = [] # Overall loss function per iteration 
    overall_loss_history = [] # Overall loss function history pre epoch
    
    # Initialize an upper limit for loss (l_lim) with a large value
    l_lim = 1e+20
    
    # Generate evenly spaced points within the specified domain
    points = torch.linspace(domain[0], domain[1], batches).view(-1, 1)
    # Enable gradient tracking for these points (useful for optimization)
    points.requires_grad = True
    # Create a data loader for the points with the specified batch size
    loader = DataLoader(points, batch_size,shuffle=True)

    # Define the initial time point t0 and enable gradient tracking for this point
    t0 = torch.tensor([0.0]).view(-1, 1)
    t0.requires_grad = True
    
    # Compute initial function value y0 and its derivative dy0_dt
    y0 = Solution(t0, params).view(-1, 1)
    dy0_dt = grad(y0, t0)

    # Record the start time before training begins
    time_i = time.time()

    # Training loop over epochs
    for epoch in range(epochs):
        # Iterate through data loader batches
        for t in loader:
            # Concatenate t with the initial time point t0
            t = torch.cat((t, t0))
            # Iterate over each batch
            for itr in range(num_itr):
                # Pass the input data (feature) through the neural network and obtain alpha and beta
                y = model(t)
                
                # Compute physics-informed and initial conditions losses, then combine them
                dy_dt = grad(y,t) # Compute first derivative of alpha with respect to r
                d2y_dt2 = grad(dy_dt, t) # Compute second derivative of alpha with respect to r
                phy_l= phy_loss(t, y, dy_dt, d2y_dt2, params)
                bc1_l = bc1_loss(y0, y[-1].view(-1, 1))
                bc2_l = bc2_loss(dy0_dt, dy_dt[-1].view(-1, 1))
                tot_l = loss_weights[0]*phy_l + loss_weights[1]*bc1_l + loss_weights[2]*bc2_l
                
                # Update model parameters: zero gradients, compute backward pass, and perform optimization step
                optimizer.zero_grad()
                tot_l.backward(retain_graph=True)
                optimizer.step()
                
                # Record overall loss function for every iteration
                loss_history.append(tot_l.item())

        # Compute losses and various expressions to record them
        y = model(points)
        dy_dt = grad(y,points)
        d2y_dt2 = grad(dy_dt, points)
        phy_l= phy_loss(t, y, dy_dt, d2y_dt2, params)
        bc1_l = bc1_loss(y0, y[0].view(-1, 1))
        bc2_l = bc2_loss(dy0_dt, dy_dt[0].view(-1, 1))
        tot_l = loss_weights[0]*phy_l + loss_weights[1]*bc1_l + loss_weights[2]*bc2_l

        # Append each losses and various expressions to the respective lists
        phy_loss_history.append(phy_l.item())
        bc1_loss_history.append(bc1_l.item())
        bc2_loss_history.append(bc2_l.item())
        overall_loss_history.append(tot_l.item())
        
        # Print the loss value at specific epoch during training
        print(f'epoch {epoch+1}/{epochs}, loss = {tot_l}')

        # Update the best neural network if the total loss is lower than the current limit
        if  tot_l < l_lim:
            bNN =  copy.deepcopy(model)
            l_lim = tot_l

    # Collect different loss histories for analysis            
    loss_histories = [overall_loss_history, phy_loss_history, bc1_loss_history, bc2_loss_history, loss_history]

    # Find the minimum overall loss value, its iteration, and print the result
    min_loss = min(overall_loss_history)
    min_index_loss = overall_loss_history.index(min_loss)
    print(f"The minimum loss function value occurs at epoch {min_index_loss+1}: {overall_loss_history[min_index_loss]}")

    # Measure the runtime of the training process
    time_f = time.time()
    runtime = time_f - time_i
    print(f"train took {runtime} s")
    
    # Save model state dictionaries and loss histories if requested
    if save_model[0] == True:
        torch.save(bNN.state_dict(), save_model[1]+".pth")
        torch.save(loss_histories, "loss_"+save_model[1]+".pt")

    return bNN, loss_histories

# Classical Neural Network

In [ ]:
def init_weights(m):
    """
    Initializes weights and biases for a linear layer.

    Args:
        m (nn.Linear): The linear layer to initialize.

    Notes:
        - Applies uniform weight initialization within the range [-n0, n0].
        - Initializes biases with a constant value of 0.01.
    """
    if isinstance(m, nn.Linear):
        out_sz = torch.tensor(m.out_features)
        n0 = 1.0/torch.sqrt(out_sz)
        m.weight.data.uniform_(-n0, n0)
        # m.bias.data.fill_(0.01)

class CNeuralNet(nn.Module):
    """
    Classical neural network class.

    Args:
        input_size (int): Size of the input features.
        hidden_size (int): Number of hidden units in the neural network.
        output_size (int): Size of the output.

    Attributes:
        fnn (nn.Sequential): Feedforward neural network with three linear layer followed by a Silu activation.

    Notes:
        - The classical neural network architecture consists of three hidden layers.
        - The input layer has 'input_size' neurons.
        - The output layer has 'output_size' neurons.
    """
    def __init__(self, input_size, hidden_size, output_size):
        super(CNeuralNet, self).__init__()
        self.fnn = nn.Sequential(
            nn.Linear(input_size, hidden_size),
            nn.SiLU(),
            nn.Linear(hidden_size, hidden_size),
            nn.SiLU(),
            nn.Linear(hidden_size, hidden_size),
            nn.SiLU(),
            nn.Linear(hidden_size, output_size),
        )

        self.fnn.apply(init_weights)

    def forward(self, x):
        """
        Forward pass through the neural network.

        Args:
            x (torch.Tensor): Input data.

        Returns:
            torch.Tensor: Outputs predictions from the neural network.
        """
        out = self.fnn(x)
        out = out[:, 0]*out[:, 1]
        out = out.view(-1, 1)
        return out

In [ ]:
for h_size in [10, 30, 50]:
    # Set the random seed for reproducibility
    torch.manual_seed(seed)
    # Define classical neural network by input, hidden, and output sizes
    input_size , hidden_size, output_size = 1, h_size, 2
    model_classic = CNeuralNet(input_size, hidden_size, output_size)
    
    # Set optimization parameters
    betas, eps , lr = [0.9, 0.99], 1e-8, 0.001
    optimizer = optim.Adam(model_classic.parameters(), betas=betas, eps=eps, lr=lr)
    
    # Print the model architecture
    print(model_classic)
        
    # Set the necessary parameters for training and test
    A , k, m, b = 0.5, 9, 1, 1.6
    omega = np.sqrt(k/m)
    beta = b / (2*m)
    params = [A, omega, beta]
    num_itr, epochs, = 100, 20
    batches , batch_size = 124, 31
    domain, loss_weights = [0, 8], [1, 1, 1]
    
    # Initialize an empty list to store the loss history
    loss_hist_classic = []
    # Specify whether to save the model and the desired filename
    save_model = [True, f"Classical_model_{seed}_{h_size}"]
    
    # Train the physics-informed neural network (PINN)
    bmodel_classic, loss_hist_classic = train_PINN(model_classic, batches, batch_size, domain, params ,epochs, num_itr, loss_weights, seed, save_model)

# Quantum Neural Network

In [ ]:
# Set the number of qubits to 3
n_qubits = 2
# Create a quantum device with 3 qubits
dev = qml.device("default.qubit", wires=n_qubits)

def RY_layer(w):
    """
    Apply a layer of single-qubit rotations around the Y-axis (RY gates).

    Args:
        w (torch.Tensor): List of rotation angles for each qubit.
    """
    for idx, element in enumerate(w):
        qml.RY(element, wires=idx)

def entangling_layer(nqubits):
    """
    Apply an entangling layer of CNOT gates to adjacent qubits.

    Args:
        nqubits (int): Number of qubits in use.
    """
    for i in range(0, nqubits - 1):
        qml.CNOT(wires=[i, i + 1])

@qml.qnode(dev)
def quantum_net(input_features, n_qubits, ansatz_depth, feature_weights, ansatz_weights, out_weights):
    """
    Quantum circuit

    Args:
        input_features (torch.Tensor): Input features.
        n_qubits (int): Number of qubits in the circuit.
        ansatz_depth (int): Depth of the ansatz (number of layers).
        feature_weights (torch.Tensor): List of parameters
        ansatz_weights (torch.Tensor): List of parameters
        out_weights (torch.Tensor): List of parameters

    Returns:
        tuple: Tuple of expectation values for Pauli-Z operators on qubits 0 and 1 as the solution.
    """
    # apply feature map layers
    temp1 = feature_weights[0]*input_features[0]
    temp2 = feature_weights[1]*input_features[0]
    qml.exp(qml.PauliZ(0), temp1[0])
    qml.exp(qml.PauliZ(1), temp2[0])
    entangling_layer(n_qubits)
    temp3 = feature_weights[2]*(input_features[0])
    temp4 = feature_weights[3]*(input_features[0])
    qml.RY(temp3, wires=0)
    qml.RY(temp4, wires=1)
    entangling_layer(n_qubits)

    # apply ansatz layers
    for k in range(ansatz_depth):
        RY_layer(ansatz_weights[k])
        entangling_layer(n_qubits)

    # Compute expectation values for Pauli-Z operators
    exp_vals = [qml.expval(out_weights[0]*(qml.PauliZ(0)@qml.PauliZ(1)))]
    return tuple(exp_vals)


class QNeuralNet(nn.Module):
    """
    quantum neural network class.

    Args:
        n_qubits (int): Number of qubits in the circuit.
        feature_depth (int): Depth of the feature map (number of layers).
        ansatz_depth (int): Depth of the ansatz (number of layers).

    Notes:
        - The quantum neural network architecture consists of two qubits.
        - The feature map has 'feature_depth' layers.
        - The ansatz has 'ansatz_depgh' layers.
    """
    def __init__(self, n_qubits, feature_depth, ansatz_depth):
        super(QNeuralNet, self).__init__()

        # Set the number qubits, number of ansatz layer and feature map layers
        self.n_qubits = n_qubits
        self.ansatz_depth = ansatz_depth
        self.feature_depth = feature_depth
        
        # Initialize parameters of quantum circuit
        self.feature_weights = nn.Parameter(0.1*torch.randn((feature_depth*n_qubits + 2)).view(-1, 1))
        self.ansatz_weights = nn.Parameter(torch.randn(ansatz_depth*n_qubits).view(ansatz_depth, n_qubits))
        self.out_weights = nn.Parameter(torch.randn((1)))

    def forward(self, x):
        """
        Forward pass through the neural network.

        Args:
            x (torch.Tensor): Input data.

        Returns:
            torch.Tensor: Outputs predictions from the neural network.
        """
        # Initialize an empty tensor for quantum output
        q_out = torch.Tensor(0, 1)

        for elem in x:
            # Apply the quantum circuit (quantum_net) to each input element
            q_out_elem = torch.hstack(tuple(quantum_net(elem, self.n_qubits, self.ansatz_depth, self.feature_weights,self.ansatz_weights, self.out_weights))).float().unsqueeze(0)
            q_out = torch.cat((q_out, q_out_elem)) # Concatenate quantum outputs

        return q_out

In [ ]:
for depth in [1, 2, 3]:
    # Set the random seed for reproducibility
    torch.manual_seed(seed)
    # Define quantum neural network by number of qubits, feature layers and ansatz layers
    feature_depth, ansatz_depth = 1, depth
    model_quantum = QNeuralNet(n_qubits, feature_depth, ansatz_depth)
    
    # Set optimization parameters
    betas, eps , lr = [0.9, 0.99], 1e-8, 0.05
    optimizer = optim.Adam(model_quantum.parameters(), betas=betas, eps=eps, lr=lr)
    
    # Print the model architecture
    print(model_quantum)
    
    # Set the necessary parameters for training and test
    A , k, m, b = 0.5, 9, 1, 1.6
    omega = np.sqrt(k/m)
    beta = b / (2*m)
    params = [A, omega, beta]
    num_itr, epochs, = 100, 20
    batches , batch_size = 124, 31
    domain, loss_weights = [0, 8], [1, 1, 1]
    
    # Initialize an empty list to store the loss history
    loss_hist_classic = []
    # Specify whether to save the model and the desired filename
    save_model = [True, f"Quantum_model_14_{depth}"]
    
    # Train the physics-informed neural network (PINN)
    bmodel_quantum, loss_hist_quantum = train_PINN(model_quantum, batches, batch_size, domain, params ,epochs, num_itr, loss_weights, seed, save_model)

# Hybrid Quantum Neural Network

In [ ]:
# Set the number of qubits to 3
n_qubits = 3
# Create a quantum device with 3 qubits
dev = qml.device("default.qubit", wires=n_qubits)

def RY_layer(w):
    """
    Apply a layer of single-qubit rotations around the Y-axis (RY gates).

    Args:
        w (torch.Tensor): List of rotation angles for each qubit.
    """
    for idx, element in enumerate(w):
        qml.RY(element, wires=idx)

def entangling_layer(nqubits):
    """
    Apply an entangling layer of CNOT gates to adjacent qubits.

    Args:
        nqubits (int): Number of qubits in use.
    """
    for i in range(0, nqubits - 1):
        qml.CNOT(wires=[i, i + 1])

def init_weights(m):
    """
    Initializes weights and biases for a linear layer.

    Args:
        m (nn.Linear): The linear layer to initialize.

    Notes:
        - Applies uniform weight initialization within the range [-n0, n0].
        - Initializes biases with a constant value of 0.01.
    """
    if isinstance(m, nn.Linear):
        out_sz = torch.tensor(m.out_features)
        n0 = 1.0/torch.sqrt(out_sz)
        m.weight.data.uniform_(-n0, n0)
        # m.bias.data.fill_(0.01)

@qml.qnode(dev)
def hquantum_net(input_features, n_qubits, q_depth, q_weights):
    """
    Quantum circuit as a part of quantum hybrid neural network.

    Args:
        input_features (torch.Tensor): Input features.
        n_qubits (int): Number of qubits in the circuit.
        q_depth (int): Number of quantum layers.
        q_weights (torch.Tensor): List of parameters.
    Returns:
        tuple: Tuple of expectation values for Pauli-Z operators on qubits 0, 1, and 2.
    """

    RY_layer(input_features[0])

    entangling_layer(n_qubits)
    RY_layer(input_features[1])
    RY_layer(q_weights[0])

    entangling_layer(n_qubits)
    RY_layer(input_features[2])
    RY_layer(q_weights[1])

    entangling_layer(n_qubits)
    RY_layer(input_features[3])
    RY_layer(q_weights[2])

    exp_vals = [qml.expval(qml.PauliZ(position)) for position in range(0, n_qubits)]
    return tuple(exp_vals)

class HQNeuralNet(nn.Module):
    """
    quantum hybrid neural network class.

    Args:
        input_size (int): Size of the input features.
        hidden_size (int): Number of hidden units in the neural network.
        n_qubits (int): Number of qubits in the circuit.
        q_depth (int): Number of quantum layers.
        output_size (int): Size of the output.

    Attributes:
        fnn (nn.Sequential): Feedforward neural network with a linear layer followed by a Tanh activation.

    Notes:
        - The first classical neural network architecture consists of one hidden layer.
        - The input layer has 'input_size' neurons.
        - The hidden layers has 'hidden_size' neurons with a Tanh activation.
        - The quantum neural network architecture consists of 'n_qubits' and 'q_depth ' layers.
        - The second classical neural network architecture consists of one linear layer.
        - The output layer has 'output_size' neurons.
    """
    def __init__(self, input_size, hidden_size, n_qubits, q_depth, output_size):
        super(HQNeuralNet, self).__init__()
        
        # Define the feedforward neural network (fnn)
        self.fnn = nn.Sequential(
            nn.Linear(input_size, hidden_size),
            nn.SiLU(),
            nn.Linear(hidden_size, (q_depth+1)*n_qubits),
        )
        self.fnn.apply(init_weights)

        #Define the feedforward (n_qubits, output_size) layer as a output layer
        self.output_layer = nn.Linear(n_qubits, output_size)
        self.output_layer.apply(init_weights)

        # Set the number qubits, number of quantum layers
        self.n_qubits = n_qubits
        self.q_depth = q_depth
        
        # Initialize parameters of quantum circuit
        self.q_weights = nn.Parameter(0.2*torch.randn(q_depth * n_qubits).view(q_depth, n_qubits))

    def forward(self, x):
        """
        Forward pass through the neural network.

        Args:
            data (torch.Tensor): Input data.

        Returns:
            torch.Tensor: Outputs predictions from the neural network.
        """
        # Pass the data through the feedforward neural network (self.fnn)
        out = self.fnn(x)
        # Initialize an empty tensor for quantum output
        q_out = torch.Tensor(0, self.n_qubits)
        
        for elem in out:
            # Apply the quantum circuit (hquantum_net) to each output element
            elem = elem.view((self.q_depth+1), self.n_qubits)
            q_out_elem = torch.hstack(tuple(hquantum_net(elem, self.n_qubits, self.q_depth, self.q_weights))).float().unsqueeze(0)
            q_out = torch.cat((q_out, q_out_elem))
            
        # Pass the output of quantum circuit through the feedforward linear layer then combine outputs
        out = self.output_layer(q_out)
        out1 = out[:, 0].view(-1, 1)
        out2 = out[:, 1].view(-1, 1)
        return out1*out2

In [ ]:
for h_size in [10, 30, 50]:
    # Set the random seed for reproducibility
    torch.manual_seed(seed)
    # Define input, hidden, and output sizes and specify the domain (range) for input data
    input_size, hidden_size, q_depth, output_size = 1, h_size, 3, 2
    model_hybrid = HQNeuralNet(input_size, hidden_size, n_qubits, q_depth, output_size)
    
    # Create a hybrid quantum neuralnet class and print the model architecture
    betas, eps , lr = [0.9, 0.99], 1e-8, 0.001
    optimizer = optim.Adam(model_hybrid.parameters(), betas=betas, eps=eps, lr=lr)
    
    # Print the model architecture
    print(model_hybrid)
    
    # Set the necessary parameters for training and test
    A , k, m, b = 0.5, 9, 1, 1.6
    omega = np.sqrt(k/m)
    beta = b / (2*m)
    params = [A, omega, beta]
    num_itr, epochs, = 100, 20
    batches , batch_size = 124, 31
    domain, loss_weights = [0, 8], [1, 1, 1]
    
    # Initialize an empty list to store the loss history
    loss_hist_classic = []
    # Specify whether to save the model and the desired filename
    save_model = [True, f"hybrid_model_{seed}_{h_size}"]
    
    # Train the physics-informed neural network (PINN)
    bmodel_hybrid, loss_hist_hybrid = train_PINN(model_hybrid, batches, batch_size, domain, params ,epochs, num_itr, loss_weights, seed, save_model)